# Visual defect classification
HOG feature extraction and an RBF-SVM baseline. Raw images and labels must be supplied under `part1_inspection/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

DATA = Path('part1_inspection')
LABELS = pd.read_csv(DATA / 'train' / 'labels.csv')
TRAIN_DIR = DATA / 'train'
train_images = {p.stem:p for p in TRAIN_DIR.rglob('*.png')}


In [ ]:
IMAGE_SIZE = (128, 128)
def extract_visual_features(path):
    with Image.open(path) as image:
        image = image.convert('L').resize(IMAGE_SIZE)
        array = np.asarray(image, dtype=np.float32) / 255.0
    return hog(array, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), block_norm='L2-Hys')

unit_ids = LABELS.unit_id.astype(str).tolist()
missing = [uid for uid in unit_ids if uid not in train_images]
if missing:
    raise FileNotFoundError(f'Missing {len(missing)} training images; examples: {missing[:5]}')
X = np.vstack([extract_visual_features(train_images[uid]) for uid in unit_ids])
y = LABELS.visual_label.astype(str).str.upper().eq('NG').astype(int).to_numpy()
print('Feature matrix:', X.shape, 'NG:', int(y.sum()))


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model = make_pipeline(StandardScaler(), SVC(kernel='rbf', class_weight='balanced',
                                             probability=True, random_state=42))
model.fit(X_train, y_train)
scores = model.predict_proba(X_val)[:, 1]
pred = (scores >= 0.10).astype(int)
print(confusion_matrix(y_val, pred, labels=[0, 1]))
print(classification_report(y_val, pred, target_names=['OK', 'NG'], zero_division=0))
print('ROC-AUC:', roc_auc_score(y_val, scores))
